# Module 07: OpenLineage & Lineage UI

## What You'll Learn

- What OpenLineage is and why lineage matters
- How Feast emits OpenLineage events
- Configuring Feast for lineage tracking
- Marquez as the lineage backend
- Visualizing lineage graphs
- Cross-system lineage (Feast + Spark + KFP)

---

> **🗺️ DATA STRATEGY**: This is **Workshop Decision #1** — E2E Lineage via OpenLineage. The highest-priority decision from the Jan 2026 workshop. Full data audit trail: source → transformations → model → agent. Feast is the first production OpenLineage emitter in the RHOAI stack.

## What Is OpenLineage?

**OpenLineage** is an open standard (LFAI graduate project) for collecting and analyzing lineage metadata. It defines a common format for data lineage events that any tool can emit.

```
Producers (emit events):          Consumer (stores & visualizes):
┌─────────────────────┐       ┌────────────────┐
│ Feast (apply/materialize) │─────▶│                │
└─────────────────────┘       │    Marquez     │
┌─────────────────────┐       │  (lineage DB  │
│ Spark (jobs)              │─────▶│   + UI)       │
└─────────────────────┘       │                │
┌─────────────────────┐       │                │
│ KFP (pipeline runs)       │─────▶│                │
└─────────────────────┘       └────────────────┘
```

**Why lineage matters:**
- **Compliance**: Gartner MQ explicitly asked Red Hat about lineage capabilities
- **Debugging**: "Why did my model's performance degrade?" → trace back to data source changes
- **Trust**: Regulated industries require full audit trails
- **Impact analysis**: "If I change this data source, what models are affected?"

## Feast OpenLineage Events

Feast emits OpenLineage events at two points:

| Operation | Event | What It Records |
|-----------|-------|----------------|
| `feast apply` | RunEvent (START/COMPLETE) | Schema changes: data source → feature view relationships |
| `feast materialize` | RunEvent (START/COMPLETE) | Data movement: offline store → online store |

### Event Structure

```json
{
  "eventType": "COMPLETE",
  "eventTime": "2024-03-15T10:30:00Z",
  "run": {"runId": "abc-123"},
  "job": {
    "namespace": "feast",
    "name": "credit_scoring.materialize.credit_history"
  },
  "inputs": [{
    "namespace": "feast",
    "name": "credit_scoring.credit_timeseries_source",
    "facets": {"schema": {"fields": [...]}}
  }],
  "outputs": [{
    "namespace": "feast",
    "name": "credit_scoring.credit_history",
    "facets": {"schema": {"fields": [...]}}
  }]
}
```

## Enabling OpenLineage in Feast

Add to `feature_store.yaml`:

```yaml
project: credit_scoring
# ... other config ...

lineage:
  enabled: true
  backend_url: http://marquez-api.lineage-ns.svc.cluster.local:5000
```

That's it. No code changes required — events emit automatically.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import yaml

# Setup feature store with OpenLineage enabled
os.makedirs("data", exist_ok=True)
os.makedirs("feature_repo", exist_ok=True)

# Generate data
np.random.seed(42)
records = []
for cid in range(1, 51):
    for week in range(26):
        ts = datetime(2024, 1, 1) + timedelta(weeks=week)
        records.append({
            "customer_id": cid,
            "event_timestamp": ts,
            "credit_score": np.random.randint(300, 850),
            "transaction_count": np.random.randint(0, 500),
        })
pd.DataFrame(records).to_parquet("data/lineage_demo.parquet")

# Config with lineage enabled
# NOTE: In a real cluster, backend_url points to Marquez API service
config = {
    "project": "lineage_demo",
    "provider": "local",
    "registry": {"registry_type": "sql", "path": "sqlite:///data/registry.db"},
    "offline_store": {"type": "duckdb"},
    "online_store": {"type": "sqlite", "path": "data/online.db"},
    "entity_key_serialization_version": 3,
    # OpenLineage configuration:
    # Uncomment when Marquez is available on your cluster
    # "lineage": {
    #     "enabled": True,
    #     "backend_url": "http://marquez-api.lineage-ns.svc:5000",
    # },
}

with open("feature_repo/feature_store.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("Feature store configured")
print("\nTo enable lineage, uncomment the 'lineage' section and deploy Marquez.")
print("See manifests/ folder for Marquez deployment resources.")

In [ ]:
from feast import Entity, FeatureView, Field, FileSource, FeatureStore
from feast.types import Int64

customer = Entity(name="customer", join_keys=["customer_id"])
source = FileSource(
    name="credit_source",
    path=os.path.abspath("data/lineage_demo.parquet"),
    timestamp_field="event_timestamp",
)
credit_fv = FeatureView(
    name="credit_history",
    entities=[customer],
    ttl=timedelta(weeks=2),
    schema=[
        Field(name="credit_score", dtype=Int64),
        Field(name="transaction_count", dtype=Int64),
    ],
    source=source,
)

store = FeatureStore(repo_path="feature_repo")

# This `apply` would emit OpenLineage START + COMPLETE events
# when lineage is enabled
store.apply([customer, source, credit_fv])
print("✅ feast apply complete")
print("\nWith lineage enabled, this emits an OpenLineage RunEvent:")
print("  Job: lineage_demo.apply")
print("  Inputs: [] (schema definition, no data read)")
print("  Outputs: [credit_source, credit_history] (registered artifacts)")

In [ ]:
# Materialize - this also emits lineage events
store.materialize(
    start_date=datetime(2024, 1, 1),
    end_date=datetime.now(),
)
print("✅ feast materialize complete")
print("\nWith lineage enabled, this emits:")
print("  Job: lineage_demo.materialize.credit_history")
print("  Inputs: [credit_source] (offline store data)")
print("  Outputs: [credit_history_online] (online store)")
print("\nThe lineage graph now shows:")
print("  credit_source --> [materialize] --> credit_history (online)")

## 🖥️ Lineage UI: Marquez

Marquez provides a web UI to visualize the lineage graph.

### Deploying Marquez on RHOAI

```bash
# Deploy Marquez in a dedicated namespace
oc new-project lineage-system

# Apply Marquez manifests (see manifests/marquez/ in this repo)
oc apply -f manifests/marquez/

# Verify
oc get pods -n lineage-system
# marquez-api-xxx       Running
# marquez-web-xxx       Running  
# marquez-db-xxx        Running

# Access the UI
oc get routes -n lineage-system | grep marquez-web
```

### What You See in Marquez UI

1. **Jobs tab**: Lists all Feast operations (apply, materialize) with run history
2. **Datasets tab**: All data sources and feature views as versioned datasets
3. **Lineage graph**: Visual DAG showing data flow between datasets via jobs
4. **Run details**: Duration, status, input/output datasets for each run

> **⚠️ GAP (Marquez)**: 
> - v0.50.0 since Oct 2024 (19-month release gap) — project health concern
> - Zero built-in authentication — anyone with network access can read/write
> - Single-maintainer concentration (58% of commits)
> - UI is not embeddable — standalone webpack app, cannot integrate into RHOAI dashboard
>
> **🗺️ DATA STRATEGY**: The POC (Scenario B Phase 1) evaluates Marquez. Fallback options: Egeria or DataHub. The long-term vision is unified lineage visualization in the RHOAI Dashboard, not a standalone Marquez UI.

## Lineage Scope and Limitations

### What Feast Lineage Covers

| Event | Captured | Details |
|-------|----------|--------|
| Schema registration | ✅ | `feast apply` records data source → feature view edges |
| Materialization | ✅ | Data movement from offline → online store |
| Feature versioning | ✅ | v0.62.0 adds version indicators to lineage graph nodes |

### What's NOT Covered

| Gap | Why | Workaround |
|-----|-----|------------|
| Model training consumption | Feast is emit-only, not a lineage system | MLflow should emit OL events (upstream issue #15891) |
| Inference serving access | No runtime lineage | Future: agent lineage RFC |
| Column-level lineage | Feature view granularity only | OpenLineage column-level facets exist in spec but Feast doesn't emit them |
| Point-in-time queries | "What was the graph on date X?" | Marquez capability, not Feast |

> **⚠️ GAP**: Feast is a lineage **producer** only. It cannot receive, store, or visualize lineage. The lineage chain currently breaks at the feature store boundary — no tracking into model training or serving. Cross-system lineage requires MLflow, Spark, and KFP all emitting to the same Marquez instance.

## The E2E Lineage Vision

```
Raw Data (PostgreSQL/S3)
    │ [Spark job emits OL event]
    ▼
ETL / Transformation
    │ [KFP pipeline step emits OL event]
    ▼
Feast Data Source
    │ [feast apply emits OL event]
    ▼
Feature View
    │ [feast materialize emits OL event]
    ▼
Online Store
    │ [MLflow should emit OL event - NOT YET]
    ▼
Model Training
    │ [future: serving lineage]
    ▼
Model Inference / Agent Decision
```

Today, only the Feast portion emits automatically. The rest requires:
- Spark: OpenLineage listener (available, needs config)
- KFP: OpenLineage integration (tested in Scenario B prototype)
- MLflow: upstream issue #15891 (not emitting yet)
- Ray: manual instrumentation via openlineage-python client

In [ ]:
# Programmatic lineage event emission (for components that don't auto-emit)
# This is how you'd manually emit lineage from Ray jobs or custom code

from openlineage.client import OpenLineageClient
from openlineage.client.run import RunEvent, RunState, Run, Job
from openlineage.client.facet import (
    SchemaDatasetFacet, SchemaField
)
import uuid

# Example: manually emit a lineage event for a Ray data processing job
# (This is what you'd do for Ray jobs that aren't auto-instrumented)

print("Example: Manual OpenLineage emission")
print("")
print("from openlineage.client import OpenLineageClient")
print("")
print('client = OpenLineageClient(url="http://marquez-api:5000")')
print("")
print("# Emit at start of job")
print('client.emit(RunEvent(eventType=RunState.START, ...))')
print("")
print("# ... do work ...")
print("")
print("# Emit at completion")
print('client.emit(RunEvent(eventType=RunState.COMPLETE, ...))')
print("")
print("This is required for Ray Data jobs on RHOAI since")
print("open-source Ray has no native OpenLineage integration.")

## Key Takeaways

1. **OpenLineage is the standard** for data lineage — LFAI graduate project, widely adopted
2. **Feast emits automatically** at `apply` (schema) and `materialize` (data movement)
3. **One config line** enables it: `lineage: {enabled: true, backend_url: ...}`
4. **Marquez** is the lineage backend for collection and visualization (with project health risks)
5. **Lineage stops at Feast boundary** — cross-system lineage needs all producers emitting
6. **Workshop Decision #1** — this is the highest-priority data strategy investment

## What's Next

- **Module 08**: RAG & Vector Search — embedding features with lineage tracking
- **Module 11**: KFP Integration — pipeline-level lineage
- **Module 13**: UI & Observability — the Marquez UI in detail